In [1]:
!pip install "datasets<3.0.0" soundfile

!pip install transformers accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
!pip install resemblyzer  # lightweight speaker embedding library


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 38.5 MB/s eta 0:00:00
  Created wheel for webrtcvad: filename=webrtcvad-2.0.10-cp312-cp312-linux_x86_64.whl size=73514 sha256=fd775812ecf51409c6a9482c9aed54e21d1ab9a2ef086786435fe7ed263ef517
  Stored in directory: /root/.cache/pip/wheels/1e/d3/95/680fa3b16848f1a58d2edaed34c496224c89a9bc63e17b3614
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26304 sha256=d83a9b48966b87a4991bfd4abd21fce16e203462ec8c2262d01e23c6106e8ec7
  Stored in directory: /root/.cache/pip/wheels/12/98/52/2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built webrtcvad typing


In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from datasets import load_dataset

# Target Urdu directory
ds_urduspeech_stream = load_dataset(
    "humairawan/UrduSpeech",
    data_dir="Urdu",
    split="train",
    token='',
    streaming=True
)

# Take first NUM_SAMPLES with style = "WIKI"
urduspeech_samples = []
for sample in ds_urduspeech_stream:
    if sample.get("style") == "WIKI" and sample.get("gender") == "Male":  # filter formal
        urduspeech_samples.append(sample)
        print('Sample no.: ',len(urduspeech_samples))
    if len(urduspeech_samples) >= 30:
        break # first sample

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Sample no.:  1
Sample no.:  2
Sample no.:  3
Sample no.:  4
Sample no.:  5
Sample no.:  6
Sample no.:  7
Sample no.:  8
Sample no.:  9
Sample no.:  10
Sample no.:  11
Sample no.:  12
Sample no.:  13
Sample no.:  14
Sample no.:  15
Sample no.:  16
Sample no.:  17
Sample no.:  18
Sample no.:  19
Sample no.:  20
Sample no.:  21
Sample no.:  22
Sample no.:  23
Sample no.:  24
Sample no.:  25
Sample no.:  26
Sample no.:  27
Sample no.:  28
Sample no.:  29
Sample no.:  30


In [5]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(0,10):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Loaded the voice encoder model on cpu in 0.03 seconds.
Sample 0: آشوب چشم کی شناخت آنکھ کی جھلی کی سوزش س...
Sample 1: جَبہَۃُ الاَکْراد شام کی خانہ جنگی میں ش...
Sample 2: ایسا لگتا ہے کہ یہ نام میدانی ہاکی کی فر...
Waiting for 1.2 minutes...
Sample 3: اس رکاوٹ سے نمٹنے کے لیے، حکومت ہند جی ا...
Sample 4: سری وینکٹیشورا نیشنل پارک بھارت کے آندھر...
Sample 5: مرزا ابو الفضل حاجی علی شیرازی کے پڑ پوت...
Waiting for 1.2 minutes...
Sample 6: شائستہ خان نے علاقے سے پرتگالی اور اراکا...
Sample 7: بلیک آؤٹ کے دوران سائٹ پر جانے والے صارف...
Sample 8: غزوۂ اُحد کے بعد فتحِ مکہ تک جس قدر غزوا...
Waiting for 1.2 minutes...
Sample 9: آخر کار بھوت بھی کھڑکھڑاہٹ کے ساتھ ختم ہ...


In [6]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(10,20):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.02 seconds.
Sample 10: گھریلو طور پر ہندوستان کا زیادہ تر انحصا...
Sample 11: قرآن مجید اسلام کا اصل الاصول ہے، اس کے ...
Waiting for 1.2 minutes...
Sample 12: سنہ دو ہزار نو عیسوی میں انہیں اسلام آبا...
Sample 13: حضرت ابو درداء رضی اللہ تعالیٰ عنہ کا شم...
Sample 14: فلائیٹ لیفٹیننٹ غلام مرتضٰی ملک شہید ملک...
Waiting for 1.2 minutes...
Sample 15: زؤچوان کاؤنٹی لاطینی چین کا ایک عوامی جم...
Sample 16: اس مسلئہ سے بآسانی یہ کلیہ اخذ کیا جا سک...
Sample 17: ہرش اور پُلکیشن کے درمیان جنگ کی تاریخ پ...
Waiting for 1.2 minutes...
Sample 18: ایکالینن کا مرکز جھیل کُروس یروی کے کنار...
Sample 19: کشیارا کو دریائے کالنی کے نام سے بھی جان...


In [7]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(20,30):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.02 seconds.
Sample 20: عبد الالہ بن علی عربی عبد الإلہ شاہ حجاز...
Waiting for 1.2 minutes...
Sample 21: کنڑ ادب کے تین جواہر آدی کَوی پمپا، سری...
Sample 22: بنگلورو کے بانی کیمپے گوڈا نے سولہویں صد...
Sample 23: اجناس کی بڑھتی ہوئی قیمتیں، بے روزگاری، ...
Waiting for 1.2 minutes...
Sample 24: خشک سالی بھی نہیں سردی ہی پڑتی ہے اور كس...
Sample 25: جینوم کی مدد سے افزائش نسل کے طریقوں کا ...
Sample 26: مکندرا ہلز نیشنل پارک کوہِسْتانی ہے اور ...
Waiting for 1.2 minutes...
Sample 27: اتّر پردیش ہریانہ گجرات بہار مہاراشٹر...
Sample 28: شہدائے ھفت تیر میٹرو اسٹیشن...
Sample 29: کالج کیمپس کے قریب کفایتی تھرفٹ اسٹوروں ...
Waiting for 1.2 minutes...


In [8]:
# ============================================
# PHASE 2: Package audio files for MUSHRA
# ============================================
import os
import zipfile

# Create folder structure webMUSHRA expects
os.makedirs("mushra_audio/reference", exist_ok=True)
os.makedirs("mushra_audio/generated_gemini_tts", exist_ok=True)


for i in range(30):
    # Copy reference
    ref_src = f"ref_{i}.wav"
    gen_src = f"gen_{i}.wav"

    # Read files
    ref_data, ref_sr = sf.read(ref_src)
    gen_data, gen_sr = sf.read(gen_src)

    # Save into structured folders
    sf.write(f"mushra_audio/reference/sample_{i}.wav", ref_data, ref_sr)
    sf.write(f"mushra_audio/generated_gemini_tts/sample_{i}.wav", gen_data, gen_sr)

# Zip everything for download
with zipfile.ZipFile("30_mushra_wiki_gemini_audio.zip", "w") as zf:
    for root, dirs, files in os.walk("mushra_audio"):
        for file in files:
            zf.write(os.path.join(root, file))

print("30_mushra_audio.zip ready for download!")

# Download in Colab
from google.colab import files
files.download("30_mushra_wiki_gemini_audio.zip")

30_mushra_audio.zip ready for download!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>